# Bayesian methods: believe, observe, update

Bayesian reasoning starts from a belief (the *prior*), confronts it with the data (the *likelihood*) and draws an updated belief from them (the *posterior*). This lab shows it on short cases, each with its own data (about 1 h, plus 35 min for the optional exercises):

1. **An online shop's conversion rate** (15 min): a Beta prior updated as visits come; maximum likelihood, maximum a posteriori and posterior mean.
2. **Recognizing a wine** (10 min): a scikit-learn naive Bayes classifier on 178 wines.
3. **Rain, sprinkler and wet grass** (15 min): inference in the course's Bayesian network, by enumeration.
4. **The hidden weather** (15 min): the course's hidden Markov model, decoded by the Viterbi algorithm.
5. **Optional — A temperature and its uncertainty** (15 min): a Gaussian process regression.
6. **Optional — Three variants of a page** (20 min): a three-armed bandit, ε-greedy versus Thompson sampling.

The exercises are independent: each one can be done without the others.

## Installation and imports

`hmmlearn`, the hidden Markov models library, isn't installed on Colab: the first cell installs it.

In [ ]:
%pip install -q hmmlearn

In [ ]:
import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from hmmlearn.hmm import CategoricalHMM
from scipy import stats
from sklearn.datasets import load_wine
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

## 1. An online shop's conversion rate

An online shop launches a new campaign and wants to know its conversion rate: the share of visits that end in a purchase. Before any visit, the marketing team has an idea of it: the past campaigns were around 20%. That is the **prior**, a Beta distribution with parameters $\alpha = 2$ and $\beta = 8$, whose mean is $\alpha / (\alpha + \beta) = 20$%.

Then the visits come: each one ends in a purchase (1) or not (0). The data is simulated: the true rate, which the shop doesn't know, is 12%.

In [ ]:
alpha, beta = 2, 8  # prior: around 20% conversion
purchases = (np.random.default_rng(130).random(500) < 0.12).astype(int)
print(f"{len(purchases)} visits, {purchases.sum()} purchases")
print("The first 10:", purchases[:10])

After $n$ purchases out of $N$ visits, the posterior is still a Beta distribution: the successes add to $\alpha$, the failures to $\beta$, that is $\mathrm{Beta}(\alpha + n, \beta + N - n)$.

*Write the function `posterior(alpha, beta, n, N)`, which returns that distribution (a `scipy.stats.beta` object). Then, after the first 10 visits, print three estimates of the conversion rate:*

- *the maximum likelihood (MLE), which looks only at the data: the observed frequency $n / N$;*
- *the maximum a posteriori (MAP): the rate where the posterior density is highest, searched on a grid of rates between 0 and 1;*
- *the posterior mean, which is also the probability that the next visit ends in a purchase (the predictive).*

*What does the maximum likelihood conclude after 10 visits? What do the prior and the two other estimates do?*

**Useful documentation**

- [`scipy.stats.beta`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.beta.html)
- [`numpy.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)

In [ ]:
def posterior(alpha, beta, n, N):
  # Your code here
  ...


# Your code here

### Additional hints

- `stats.beta(a, b)` creates the Beta distribution with parameters `a` and `b`; its method `.pdf(x)` gives the density at each rate of the array `x`, and `.mean()` its mean.
- The first 10 visits: `purchases[:10]`. `n` is their sum (the number of purchases), `N` their number.
- MAP: `grid = np.linspace(0, 1, 1001)`, 1,001 rates from 0 to 1; `np.argmax(distribution.pdf(grid))` gives the position of the density's highest point, and `grid[...]` the matching rate.
- Expected values: 3 purchases out of the first 10 visits, so an MLE of 30.0%, a MAP of 22.2% and a posterior mean of 25.0%.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def posterior(alpha, beta, n, N):
  return stats.beta(..., ...)


N = 10
n = purchases[:N].sum()
distribution = posterior(alpha, beta, n, N)
grid = np.linspace(0, 1, 1001)
print(f"MLE: {...:.1%}")
print(f"MAP: {grid[np.argmax(...)]:.1%}")
print(f"Posterior mean: {...:.1%}")
```

### Solution

In [ ]:
def posterior(alpha, beta, n, N):
  return stats.beta(alpha + n, beta + N - n)


N = 10
n = purchases[:N].sum()
distribution = posterior(alpha, beta, n, N)
grid = np.linspace(0, 1, 1001)
print(f"MLE: {n / N:.1%}")
print(f"MAP: {grid[np.argmax(distribution.pdf(grid))]:.1%}")
print(f"Posterior mean: {distribution.mean():.1%}")

3 purchases out of 10 visits: the maximum likelihood concludes to a 30% conversion rate, on the strength of 10 visits only. The prior holds this hasty conclusion back: the MAP (22.2%) and the posterior mean (25%) stay between the prior (20%) and the data (30%).

### The posterior as visits come

The next cell plots the prior, then the posterior after 10, 50 and 500 visits, and prints the three estimates at each step.

In [ ]:
grid = np.linspace(0, 1, 1001)
fig, ax = plt.subplots(figsize=(8, 4))
for N in [0, 10, 50, 500]:
  n = purchases[:N].sum()
  distribution = posterior(alpha, beta, n, N)
  density = distribution.pdf(grid)
  label = "Prior" if N == 0 else f"After {N} visits ({n} purchases)"
  ax.plot(grid, density, label=label)
  if N > 0:
    print(
      f"{N} visits: MLE {n / N:.1%}, MAP {grid[np.argmax(density)]:.1%}, "
      f"posterior mean {distribution.mean():.1%}"
    )
ax.axvline(0.12, color="black", linestyle="--", label="True rate (12%)")
ax.set_xlim(0, 0.5)
ax.set_xlabel("Conversion rate")
ax.set_ylabel("Density")
ax.legend()
plt.show()

Each curve says where the rate probably is: the narrower it is, the surer the belief. With few visits, the curve is wide and the prior weighs as much as the data; with 500, it narrows around the true rate, and the three estimates meet: when data abounds, the prior hardly counts any more.

## 2. Recognizing a wine: a naive Bayes classifier

The *Wine* dataset, shipped with scikit-learn, describes 178 wines from the same region of Italy, made from three grape varieties (numbered 0, 1 and 2): 13 chemical measurements per wine (alcohol, acidity, color intensity…). The goal: find a wine's variety from its measurements. 30% of the wines are set aside for the test.

In [ ]:
wines = load_wine(as_frame=True)
X, y = wines.data, wines.target
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.3, stratify=y, random_state=0
)
print(f"{len(X_train)} training wines, {len(X_test)} test wines")
X.head()

The naive Bayes classifier picks the most probable class given the measurements, assuming them independent given the class. The course estimates its probabilities by counting; for continuous measurements like these, `GaussianNB` describes each measurement of each class by a bell curve (a normal distribution), whose mean and variance it learns.

*Train a `GaussianNB` on the training set and print its accuracy on the test set. Then print, with `predict_proba`, the probability it gives each variety for the first five test wines. Does the model hesitate?*

**Useful documentation**

- [`sklearn.naive_bayes.GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)
- [`GaussianNB.predict_proba`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html#sklearn.naive_bayes.GaussianNB.predict_proba)

In [ ]:
# Your code here

### Additional hints

- `GaussianNB()` has no parameter to tune: `.fit(X_train, y_train)`, then `.score(X_test, y_test)`, which gives the accuracy (the share of right answers).
- `model.predict_proba(X_test.head())` returns an array with one row per wine and one column per variety, each row summing to 1; it reads better in a rounded `pd.DataFrame`.
- Expected accuracy: 96.3%.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model = GaussianNB()
model.fit(..., ...)
print(f"Accuracy: {model.score(..., ...):.1%}")
probas = model.predict_proba(...)
pd.DataFrame(probas, columns=["Variety 0", "Variety 1", "Variety 2"]).round(3)
```

### Solution

In [ ]:
model = GaussianNB()
model.fit(X_train, y_train)
print(f"Accuracy: {model.score(X_test, y_test):.1%}")
probas = model.predict_proba(X_test.head())
pd.DataFrame(probas, columns=["Variety 0", "Variety 1", "Variety 2"]).round(3)

96.3% right answers, with a model that trains in an instant. But it hardly ever hesitates: its probabilities are close to 0 or 1, even when it is wrong (one of the two misclassified test wines gets 91.5% for the wrong variety).

### What the model learned

For each variety, the histogram of the training wines' alcohol, and the bell curve the model drew from it.

In [ ]:
feature = "alcohol"
j = list(X.columns).index(feature)
x = np.linspace(X[feature].min(), X[feature].max(), 200)
fig, ax = plt.subplots(figsize=(8, 4))
for variety in model.classes_:
  values = X_train.loc[y_train == variety, feature]
  ax.hist(values, bins=10, density=True, alpha=0.3, color=f"C{variety}")
  curve = stats.norm(model.theta_[variety, j], np.sqrt(model.var_[variety, j]))
  ax.plot(x, curve.pdf(x), color=f"C{variety}", label=f"Variety {variety}")
ax.set_xlabel("Alcohol (% vol.)")
ax.set_ylabel("Density")
ax.legend()
plt.show()

The whole model fits in 3 × 13 curves like these: one per variety and per measurement. To classify a wine, it multiplies the heights of its 13 curves, as if the measurements were independent. Yet several measurements say the same thing (phenols and flavonoids go together): the model counts the same clue several times, hence too clear-cut probabilities. It classifies well, but its probabilities aren't to be taken at face value.

## 3. Rain, sprinkler and wet grass

The course's Bayesian network: rain $R$ influences the sprinkler $A$ (it is seldom turned on when it rains), and both wet the grass $H$. Each variable is 1 (true) or 0 (false). The next cell takes the tables of the course's figure and draws the joint distribution from them, the product of the tables: $P(R, A, H) = P(R)\,P(A \mid R)\,P(H \mid A, R)$, one row per combination of the three variables.

In [ ]:
# The course's tables: P(R = 1), P(A = 1 | R) and P(H = 1 | A, R)
p_rain = 0.2
p_sprinkler = {0: 0.4, 1: 0.01}  # depending on R
p_grass = {(0, 0): 0.0, (0, 1): 0.8, (1, 0): 0.9, (1, 1): 0.99}  # depending on (A, R)


# P(X = value) for a variable that is 1 with probability p
def bernoulli(p, value):
  return p if value == 1 else 1 - p


rows = []
for R, A, H in itertools.product([1, 0], repeat=3):
  p = bernoulli(p_rain, R) * bernoulli(p_sprinkler[R], A) * bernoulli(p_grass[A, R], H)
  rows.append({"R": R, "A": A, "H": H, "P": p})
joint = pd.DataFrame(rows)
joint

Inferring means computing the probability of a variable given what is observed. By enumeration: keep the rows of the joint that agree with the observations, and divide the probability of those where the variable sought is 1 by their total.

*Write the function `infer(joint, target, observations)`, which returns $P(\text{target} = 1 \mid \text{observations})$, `observations` being a dictionary such as `{"H": 1}`. The cell after the solution asks it three questions.*

**Useful documentation**

- [Boolean indexing in pandas](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing)

In [ ]:
def infer(joint, target, observations):
  # Your code here
  ...

### Additional hints

- A boolean mask, true for every row to start with: `mask = pd.Series(True, index=joint.index)`. For each `variable, value` of `observations.items()`, `mask &= joint[variable] == value` keeps true only the rows that agree with the observation.
- `kept = joint[mask]`: the rows kept. The numerator is the sum of their `P` column where `target` is 1, the denominator the sum of their whole `P` column.
- Example to check: with no observation, `infer(joint, "R", {})` is 0.2, the prior of rain.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def infer(joint, target, observations):
  mask = pd.Series(True, index=joint.index)
  for variable, value in observations.items():
    mask &= ...
  kept = joint[mask]
  return kept.loc[..., "P"].sum() / kept["P"].sum()
```

### Solution

In [ ]:
def infer(joint, target, observations):
  mask = pd.Series(True, index=joint.index)
  for variable, value in observations.items():
    mask &= joint[variable] == value
  kept = joint[mask]
  return kept.loc[kept[target] == 1, "P"].sum() / kept["P"].sum()

### Three questions to the network

How probable is it that it rained, knowing nothing, then seeing the wet grass, then learning as well that the sprinkler was on?

In [ ]:
questions = {
  "P(R = 1)": {},
  "P(R = 1 | H = 1)": {"H": 1},
  "P(R = 1 | H = 1, A = 1)": {"H": 1, "A": 1},
}
for question, observations in questions.items():
  print(f"{question} = {infer(joint, 'R', observations):.3f}")

The wet grass raises the probability of rain from 20% to 35.8%: reasoning goes back from an effect to one of its possible causes, as a diagnosis does. Learning that the sprinkler was on brings it back down to 0.7%: the sprinkler alone explains the wet grass, and it is seldom turned on when it rains.

Enumeration goes through the whole joint, $2^n$ rows for $n$ binary variables: beyond a few dozen variables, Bayesian network libraries (`pgmpy`, for instance) compute without building the joint, following the graph's structure, or approximate the result by sampling.

## 4. The hidden weather: the Viterbi algorithm

The course's hidden Markov model: each day, the pressure is high ($H$) or low ($L$), unseen; only the weather is observed, Sunny, Cloudy or Rainy. Its parameters are those of the course's figure: the start probabilities $\pi$, the transitions $T$ (row: the previous day's state, column: the day's) and the emissions $E$ (row: the state, column: the observation).

In [ ]:
states = ["H", "L"]  # high and low pressure
observations = ["Sunny", "Cloudy", "Rainy"]
pi = np.array([0.7, 0.3])
T = np.array([[0.3, 0.7], [0.4, 0.6]])
E = np.array([[0.6, 0.1, 0.3], [0.2, 0.4, 0.4]])

# Five observed days, each coded by its position in `observations`, as a column
days = ["Sunny", "Cloudy", "Cloudy", "Rainy", "Sunny"]
sequence = np.array([[observations.index(day)] for day in days])
sequence.ravel()

*Create a two-state `CategoricalHMM` and give it these parameters (its attributes `startprob_`, `transmat_` and `emissionprob_`). Then find, with its `decode` method, the most probable sequence of hidden states for these five days, and print it with the states' names. Is it the course's?*

**Useful documentation**

- [`hmmlearn.hmm.CategoricalHMM`](https://hmmlearn.readthedocs.io/en/latest/api.html#hmmlearn.hmm.CategoricalHMM)
- [`CategoricalHMM.decode`](https://hmmlearn.readthedocs.io/en/latest/api.html#hmmlearn.hmm.CategoricalHMM.decode)

In [ ]:
# Your code here

### Additional hints

- `CategoricalHMM(n_components=2)` creates a two-state model. Its parameters are set directly (`weather_model.startprob_ = pi`, likewise `transmat_` and `emissionprob_`): nothing to train, they are known.
- `weather_model.decode(sequence, algorithm="viterbi")` returns two values: the logarithm of the best path's probability, and that path, an array of state indices (0 for H, 1 for L).
- `[states[i] for i in path]` gives the states' names; `np.exp` gives the probability back from its logarithm.
- Expected: the path (H, L, L, L, H), with probability 0.0016, the value of its last node in the course's Viterbi trellis.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
weather_model = CategoricalHMM(n_components=2)
weather_model.startprob_ = pi
weather_model.transmat_ = ...
weather_model.emissionprob_ = ...
log_proba, path = weather_model.decode(..., algorithm="viterbi")
print("Most probable states:", [states[i] for i in ...])
print(f"Probability of the path: {np.exp(...):.4f}")
```

### Solution

In [ ]:
weather_model = CategoricalHMM(n_components=2)
weather_model.startprob_ = pi
weather_model.transmat_ = T
weather_model.emissionprob_ = E
log_proba, path = weather_model.decode(sequence, algorithm="viterbi")
print("Most probable states:", [states[i] for i in path])
print(f"Probability of the path: {np.exp(log_proba):.4f}")

### Probability of each state, day by day

Viterbi gives the best path as a whole. The `predict_proba` method (the course's Forward-Backward algorithm) gives something else: for each day, the probability of each state given the whole observed sequence.

In [ ]:
table = pd.DataFrame(weather_model.predict_proba(sequence), columns=states).round(3)
table.insert(0, "Observation", days)
table

Here, each day's most probable state follows the Viterbi path, which isn't always the case: the best path as a whole isn't necessarily made of the best states taken one by one. These probabilities also tell the model's confidence: 88.5% of high pressure on the first day, only 70% of low pressure on the fourth.

## 5. Optional: a temperature and its uncertainty

*Optional exercise, for those who have finished the others.*

A sensor records a greenhouse's temperature through a day, but breaks down from 6 a.m. to 5 p.m. A Gaussian process predicts the temperature at any hour with its uncertainty: a mean and a band around it. The measurements are simulated.

In [ ]:
hours = np.array([0.5, 2, 4, 6, 17, 19, 21.5, 23.5])
noise = np.random.default_rng(0).normal(0, 0.5, hours.size)
temperatures = 14 + 6 * np.sin((hours - 9) * np.pi / 12) + noise
pd.DataFrame({"Hour": hours, "Temperature (°C)": temperatures.round(1)})

*Train a `GaussianProcessRegressor` on these measurements, predict the temperature from 0 to 24 h with its mean and standard deviation, then plot the mean, the band "mean ± 2 standard deviations" and the measurements. Where is the band widest, and why?*

**Useful documentation**

- [`sklearn.gaussian_process.GaussianProcessRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html)
- [`sklearn.gaussian_process.kernels.RBF`](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.kernels.RBF.html)
- [`sklearn.gaussian_process.kernels.WhiteKernel`](https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.kernels.WhiteKernel.html)
- [`matplotlib.axes.Axes.fill_between`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.fill_between.html)

In [ ]:
# Your code here

### Additional hints

- The kernel says what the model assumes about the curve: `RBF(length_scale=3)`, that two close hours (within about 3 h, a scale the training adjusts) have close temperatures; `+ WhiteKernel(noise_level=0.25)`, that each measurement has a little noise.
- `GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0)`: `normalize_y=True` centers the temperatures before training.
- scikit-learn wants the inputs as a column: `hours.reshape(-1, 1)` for `.fit`, likewise for the prediction grid, `np.linspace(0, 24, 241)`.
- `.predict(..., return_std=True)` returns the mean and the standard deviation at each hour of the grid; `ax.fill_between(x, low, high, alpha=0.3)` colors the band between `low` and `high`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
kernel = RBF(length_scale=3) + WhiteKernel(noise_level=0.25)
process = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0)
process.fit(..., temperatures)
hour_grid = np.linspace(0, 24, 241)
mean, std = process.predict(..., return_std=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hour_grid, mean, label="Posterior mean")
ax.fill_between(hour_grid, ..., ..., alpha=0.3, label="± 2 standard deviations")
ax.scatter(hours, temperatures, color="black", zorder=3, label="Measurements")
ax.set_xlabel("Hour")
ax.set_ylabel("Temperature (°C)")
ax.legend()
plt.show()
```

### Solution

In [ ]:
kernel = RBF(length_scale=3) + WhiteKernel(noise_level=0.25)
process = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=0)
process.fit(hours.reshape(-1, 1), temperatures)
hour_grid = np.linspace(0, 24, 241)
mean, std = process.predict(hour_grid.reshape(-1, 1), return_std=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hour_grid, mean, label="Posterior mean")
ax.fill_between(
  hour_grid,
  mean - 2 * std,
  mean + 2 * std,
  alpha=0.3,
  label="± 2 standard deviations",
)
ax.scatter(hours, temperatures, color="black", zorder=3, label="Measurements")
ax.set_xlabel("Hour")
ax.set_ylabel("Temperature (°C)")
ax.legend()
plt.show()

The band is narrow near the measurements and widens during the breakdown, from 6 a.m. to 5 p.m.: far from any measurement, the model knows it doesn't know. An ordinary regression would draw the same curve without saying where to distrust it. That is what the Bayesian approach brings: a prediction with its uncertainty.

## 6. Optional: three variants of a page, ε-greedy versus Thompson

*Optional exercise, for those who have finished the others.*

The shop of section 1 now hesitates between three versions of its product page, A, B and C. Each visit is shown one of them and ends in a purchase (reward 1) or not (0): it's a three-armed bandit. Showing the three versions in equal shares for long costs sales; showing only the one that looks best risks keeping the wrong one. Two strategies trade off exploring and exploiting:

- **ε-greedy** (ε = 0.1): one visit in ten, a version at random; the others, the version with the best rate observed so far;
- **Thompson sampling**: each version has its Beta posterior, updated as in section 1; at each visit, a rate is drawn from each one and the version with the largest draw is shown.

Their cost is measured by the **regret**: after $T$ visits, $R_T = T\mu^\star - \sum_{t=1}^{T} \mu_{k_t}$, where $\mu_{k_t}$ is the true rate of the version shown at visit $t$ and $\mu^\star$ that of the best one. It's the number of purchases lost, on average, for not always showing the best version.

The data is simulated: the true rates, which the shop doesn't know, are 10%, 12% and 15%, close enough for the first visits not to settle it. So that the comparison doesn't hang on a lucky draw, it's repeated 20 times over 10,000 visits: `results[r, t, k]` says whether visit `t` of repetition `r` would buy in front of version `k`, and a strategy only reads the entry of the version it shows.

In [ ]:
# The true conversion rates of versions A, B and C, which the shop doesn't know
true_rates = np.array([0.10, 0.12, 0.15])
n_repetitions, n_visits = 20, 10_000
# results[r, t, k]: would visit t of repetition r buy in front of version k?
shape = (n_repetitions, n_visits, 3)
results = np.random.default_rng(131).random(shape) < true_rates
print(f"{n_repetitions} repetitions of {n_visits} visits, 3 versions")

*Write the two strategies, `epsilon_greedy(results, epsilon, rng)` and `thompson(results, rng)`. Each one receives the visits of one repetition (`results[r]`, one row per visit, one column per version), chooses a version at each visit, only learns the visit's answer to that version (purchase or not) and returns the array of the versions shown. Thompson's prior is the uniform Beta(1, 1), the course's: the shop has no history of these three versions, and the same prior for all favors none.*

*Then write `cumulative_regret(choices)`, which returns each repetition's cumulative regret visit after visit, run the two strategies (ε = 0.1) on the 20 repetitions and plot their mean cumulative regret on the same chart. Which one regrets less, and why?*

**Useful documentation**

- [`numpy.random.Generator.random`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.random.html)
- [`numpy.random.Generator.integers`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.integers.html)
- [`numpy.random.Generator.beta`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.beta.html)
- [`numpy.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)
- [`numpy.cumsum`](https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html)

In [ ]:
def epsilon_greedy(results, epsilon, rng):
  # Your code here
  ...


def thompson(results, rng):
  # Your code here
  ...


# Your code here

### Additional hints

- A single seed for the whole experiment: `rng = np.random.default_rng(1)`, passed to both strategies. `rng.random() < epsilon` is true one time in ten for `epsilon=0.1`, and `rng.integers(n_versions)` then draws a version among 0, 1 and 2.
- Observed rates: `successes / np.maximum(trials, 1)`, where `np.maximum` avoids dividing by zero for a version never shown yet (it counts as 0); `np.argmax` gives the version with the best rate.
- Thompson: with the Beta(1, 1) prior, section 1's rule gives the Beta(1 + successes, 1 + failures) posterior. `rng.beta(1 + successes, 1 + failures)` draws a rate from each of the three at once (an array of parameters gives one draw per entry). `posterior(1, 1, ...)` would give the same distribution, but creating a scipy object at each visit is tens of times slower over 200,000 visits.
- Regret: `true_rates[choices]` gives the true rate of the version shown at each visit (indexing by an array of indices), and `true_rates.max() - true_rates[choices]` its gap to the best: 0.05 for A, 0.03 for B, 0 for C. `np.cumsum(..., axis=1)` accumulates it visit after visit, repetition by repetition; `.mean(axis=0)` averages the repetitions.
- The computation takes a few seconds: 2 strategies × 20 repetitions × 10,000 visits.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def epsilon_greedy(results, epsilon, rng):
  n_visits, n_versions = results.shape
  successes, trials = np.zeros(n_versions), np.zeros(n_versions)
  choices = np.zeros(n_visits, dtype=int)
  for t in range(n_visits):
    if rng.random() < ...:
      k = rng.integers(...)
    else:
      k = np.argmax(...)
    successes[k] += results[t, k]
    trials[k] += 1
    choices[t] = k
  return choices


def thompson(results, rng):
  n_visits, n_versions = results.shape
  successes, failures = np.zeros(n_versions), np.zeros(n_versions)
  choices = np.zeros(n_visits, dtype=int)
  for t in range(n_visits):
    k = np.argmax(rng.beta(..., ...))
    successes[k] += ...
    failures[k] += ...
    choices[t] = k
  return choices


def cumulative_regret(choices):
  return np.cumsum(..., axis=1)


rng = np.random.default_rng(1)
greedy_choices = np.array([epsilon_greedy(r, 0.1, rng) for r in results])
thompson_choices = np.array([thompson(r, rng) for r in results])
best = np.argmax(true_rates)
fig, ax = plt.subplots(figsize=(8, 4))
strategies = {"ε-greedy (ε = 0.1)": greedy_choices, "Thompson": thompson_choices}
for name, choices in strategies.items():
  regret = cumulative_regret(choices).mean(axis=0)
  ax.plot(np.arange(1, n_visits + 1), regret, label=name)
  print(f"{name}: regret {regret[4999]:.1f} at 5000 visits, {regret[-1]:.1f} at 10000")
  share = np.mean(choices[:, -1000:] == best)
  print(f"  best version shown in {share:.1%} of the last 1000 visits")
ax.set_xlabel("Visits")
ax.set_ylabel("Mean cumulative regret")
ax.legend()
plt.show()
```

### Solution

In [ ]:
def epsilon_greedy(results, epsilon, rng):
  n_visits, n_versions = results.shape
  successes, trials = np.zeros(n_versions), np.zeros(n_versions)
  choices = np.zeros(n_visits, dtype=int)
  for t in range(n_visits):
    if rng.random() < epsilon:
      k = rng.integers(n_versions)  # exploration: a version at random
    else:
      k = np.argmax(successes / np.maximum(trials, 1))  # the best observed rate
    successes[k] += results[t, k]
    trials[k] += 1
    choices[t] = k
  return choices


def thompson(results, rng):
  n_visits, n_versions = results.shape
  successes, failures = np.zeros(n_versions), np.zeros(n_versions)
  choices = np.zeros(n_visits, dtype=int)
  for t in range(n_visits):
    # A rate drawn from each version's Beta(1 + successes, 1 + failures) posterior
    k = np.argmax(rng.beta(1 + successes, 1 + failures))
    successes[k] += results[t, k]
    failures[k] += 1 - results[t, k]
    choices[t] = k
  return choices


def cumulative_regret(choices):
  return np.cumsum(true_rates.max() - true_rates[choices], axis=1)


rng = np.random.default_rng(1)
greedy_choices = np.array([epsilon_greedy(r, 0.1, rng) for r in results])
thompson_choices = np.array([thompson(r, rng) for r in results])
best = np.argmax(true_rates)
fig, ax = plt.subplots(figsize=(8, 4))
strategies = {"ε-greedy (ε = 0.1)": greedy_choices, "Thompson": thompson_choices}
for name, choices in strategies.items():
  regret = cumulative_regret(choices).mean(axis=0)
  ax.plot(np.arange(1, n_visits + 1), regret, label=name)
  print(f"{name}: regret {regret[4999]:.1f} at 5000 visits, {regret[-1]:.1f} at 10000")
  share = np.mean(choices[:, -1000:] == best)
  print(f"  best version shown in {share:.1%} of the last 1000 visits")
ax.set_xlabel("Visits")
ax.set_ylabel("Mean cumulative regret")
ax.legend()
plt.show()

The two curves start together, then part: after 10,000 visits, ε-greedy has lost 82.7 purchases on average, Thompson 52.2. And the gap widens: over the last 5,000 visits, ε-greedy still loses 24.1 of them, Thompson 7.5.

ε-greedy explores one visit in ten whatever it already knows: two times out of three, it then shows a worse version, which costs at least 2.7 purchases per 1,000 visits, endlessly. Its regret grows in a straight line, and it can't show the best version in more than 93.3% of the visits (90% + 10% / 3): 88.8% of the last 1,000. The rest of the time, it trusts the best observed rate, even over few visits: in one of the 20 repetitions, it showed B (12%) to 9,029 visits out of 10,000, for a regret of 294 on its own.

Thompson shows each version according to its probability of being the best. As the posteriors tighten, those of A and B almost never give the largest draw anymore: exploration dies out by itself (C gets 97.2% of the last 1,000 visits), and the curve bends, toward the course's regret in $\ln T$.

## Conclusion

- **The posterior combines the prior and the data**: with little data, the prior holds back hasty conclusions; with much, the data wins, and the maximum likelihood, the MAP and the posterior mean meet.
- **The naive Bayes classifier** trains in an instant and often classifies well, but its probabilities are too clear-cut: the independence it assumes is seldom true.
- **A Bayesian network** answers any question of the form "this cause, given these effects" from local tables.
- **A hidden Markov model** recovers unseen states from observations: the best path (Viterbi), and the confidence day by day (Forward-Backward).
- **A Gaussian process** (optional) predicts a curve and says where it is uncertain.
- **Thompson sampling** (optional) turns the posterior into a strategy: it shows each version according to its probability of being the best, its exploration dies out as the data comes in, and it regrets much less than ε-greedy, which explores endlessly.